# Анализ выживаемости: время до события с учётом цензурирования

**Проект «ИИ для учёных».** Практический блокнот к странице алгоритма «Анализ выживаемости».

## Что делает метод

Анализ выживаемости моделирует время до наступления события (отказ прибора, рецидив заболевания, прекращение наблюдения). Ключевая особенность — цензурирование: для части наблюдений известно лишь, что к концу исследования событие ещё не произошло. Игнорировать такие наблюдения нельзя — это исказит оценку.

Кривые Каплана — Мейера оценивают функцию выживания без предположений о её форме, лог-ранговый критерий сравнивает группы, а модель пропорциональных рисков Кокса оценивает влияние ковариат на риск события. В этом блокноте мы пройдём все три шага. Расчётное время прохождения — около пяти минут.

## Интуиция за методом

Представьте, что вы тестируете партию лампочек и хотите знать, сколько они служат. Вы запустили эксперимент, но через два года вынуждены его остановить: часть лампочек уже перегорела, а часть — всё ещё горит. Для тех, что ещё работают, вы знаете лишь минимальный срок службы — «не меньше стольких-то дней». Выбросить эти наблюдения нельзя: вы потеряете ценную информацию. Именно здесь нужен анализ выживаемости.

**Цензурирование** — ситуация, когда событие (перегорание лампочки, рецидив болезни, отказ прибора) к моменту окончания наблюдений не наступило. Мы знаем, что наблюдение «прожило» не меньше зафиксированного времени, но точный срок неизвестен.

**Функция выживания S(t)** — вероятность того, что событие ещё не наступило к моменту времени t. При t=0 она равна 1 (никто не пережил событие в момент начала); со временем монотонно убывает.

**Оценка Каплана — Мейера** строит эту функцию по наблюдаемым данным шагами: каждый раз, когда событие происходит у очередного объекта, кривая делает небольшой шаг вниз. Цензурированные наблюдения вклад в шаг не вносят, но уменьшают «число объектов под риском» — количество ещё не переживших событие.

**Отношение рисков (hazard ratio)** — показатель модели Кокса: во сколько раз мгновенный риск события в одной группе выше, чем в другой. Значение 2.0 означает «риск вдвое выше»; значение 0.5 — «риск вдвое ниже».

## 1. Установка библиотек

В среде Google Colab перечисленные библиотеки, как правило, уже установлены. Ячейка ниже гарантирует их наличие, в том числе при локальном запуске.

In [ ]:
%pip install -q lifelines numpy pandas matplotlib

## 2. Единый стиль графиков

Все графики в блокнотах проекта оформляются в едином визуальном стиле сайта «ИИ для учёных»: общая палитра, шрифты, толщины линий и сетка. Ниже встроено содержимое стилевого шаблона `notebooks/viz_style.py` — отдельный файл загружать не нужно. Вызов `apply_viz_style()` настраивает matplotlib один раз на весь блокнот.

In [ ]:
# Единый стилевой шаблон графиков проекта «ИИ для учёных».
# Значения синхронизированы с токенами темы сайта (styles/tokens.css).
VIZ_TOKENS = {
    "background": "#ffffff",  # фон полотна
    "node_fill":  "#eef1f6",  # заливка блоков
    "node_text":  "#0e1726",  # основной текст
    "edge":       "#4d5e78",  # линии, оси, подписи
    "grid":       "#dce2ec",  # сетка координат
    "series":     ["#2563eb", "#0d9488", "#b45309", "#4d5e78"],  # цвета рядов
}
VIZ = VIZ_TOKENS


def apply_viz_style():
    """Настраивает matplotlib под единый стиль сайта."""
    import matplotlib as mpl
    from cycler import cycler
    t = VIZ_TOKENS
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Segoe UI", "DejaVu Sans", "Arial", "Helvetica"],
        "font.size": 12, "axes.titlesize": 15, "axes.titleweight": "semibold",
        "axes.labelsize": 13, "xtick.labelsize": 11, "ytick.labelsize": 11,
        "legend.fontsize": 11,
        "figure.facecolor": t["background"], "axes.facecolor": t["background"],
        "savefig.facecolor": t["background"], "figure.dpi": 110, "savefig.dpi": 150,
        "axes.prop_cycle": cycler(color=t["series"]),
        "text.color": t["node_text"], "axes.labelcolor": t["node_text"],
        "axes.titlecolor": t["node_text"], "axes.edgecolor": t["edge"],
        "xtick.color": t["edge"], "ytick.color": t["edge"], "axes.linewidth": 1.2,
        "axes.grid": True, "grid.color": t["grid"], "grid.linewidth": 1.0,
        "grid.linestyle": "-", "axes.axisbelow": True,
        "lines.linewidth": 2.0, "lines.markersize": 6, "patch.linewidth": 1.5,
        "axes.spines.top": False, "axes.spines.right": False,
        "legend.frameon": False,
    })


def get_palette(n=None):
    """Возвращает список цветов рядов из токенов темы."""
    series = VIZ_TOKENS["series"]
    if n is None:
        return list(series)
    return [series[i % len(series)] for i in range(n)]


apply_viz_style()
print("Единый стиль графиков подключён.")

## 3. Демонстрационные данные

Используем классический набор по раку лёгкого (`load_lung` из `lifelines`): время наблюдения в днях, индикатор события (1 — произошло, 0 — цензурировано) и клинические ковариаты (возраст, пол, статус по шкале ECOG).

**Шкала ECOG** (`ecog`) оценивает общее состояние пациента от 0 (полностью активен) до 4 (прикован к постели). Это типичная ковариата в онкологических исследованиях.

Если пакет `lifelines` недоступен, ячейка автоматически генерирует синтетический набор с аналогичной структурой: здесь специально встроено цензурирование — примерно треть наблюдений не доживёт до события к моменту окончания наблюдения.

In [ ]:
import numpy as np
import pandas as pd

try:
    from lifelines.datasets import load_lung
    raw = load_lung().dropna(subset=['time', 'status', 'age',
                                     'sex', 'ph.ecog'])
    df = pd.DataFrame({
        'time': raw['time'].astype(float),
        # в наборе status: 1 - цензурирование, 2 - событие
        'event': (raw['status'] == 2).astype(int),
        'age': raw['age'].astype(float),
        'sex': raw['sex'].astype(int),
        'ecog': raw['ph.ecog'].astype(float),
    }).reset_index(drop=True)
    print('Источник данных: lifelines load_lung')
except Exception as exc:
    print('lifelines недоступна, синтетические данные:', exc)
    rng = np.random.default_rng(42)
    n = 220
    age = rng.normal(62, 9, n)
    sex = rng.integers(1, 3, n)
    ecog = rng.integers(0, 3, n).astype(float)
    risk = 0.03 * (age - 62) + 0.4 * ecog - 0.5 * (sex - 1)
    true_time = rng.exponential(np.exp(-risk) * 300)
    censor_time = rng.uniform(50, 800, n)
    df = pd.DataFrame({
        'time': np.minimum(true_time, censor_time),
        'event': (true_time <= censor_time).astype(int),
        'age': age, 'sex': sex, 'ecog': ecog})

print(f'Наблюдений: {len(df)}, '
      f'доля наступивших событий: {df["event"].mean():.0%}')
df.head()

## 4. Применение метода

### Шаг 1. Кривая Каплана — Мейера для всей выборки

Строим оценку функции выживания S(t) — вероятность не пережить событие к моменту t — без каких-либо предположений о форме этой кривой. Метод называется непараметрическим именно потому, что он не требует указывать математическую формулу кривой заранее.

Ожидаемый результат: ступенчатая кривая, убывающая от 1 к 0, с крупными шагами там, где событий много, и пологими участками там, где их мало.

In [ ]:
import matplotlib.pyplot as plt

use_lifelines = True
try:
    from lifelines import KaplanMeierFitter
    kmf = KaplanMeierFitter()
except Exception as exc:
    use_lifelines = False
    print('lifelines недоступна, ручная оценка КМ:', exc)

def km_estimate(time, event):
    """Ручная оценка Каплана - Майера (фолбэк)."""
    order = np.argsort(time)
    t, e = np.asarray(time)[order], np.asarray(event)[order]
    n_at_risk = len(t)
    surv, times, s = [], [], 1.0
    for i, ti in enumerate(t):
        if e[i] == 1:
            s *= (1 - 1 / n_at_risk)
        times.append(ti)
        surv.append(s)
        n_at_risk -= 1
    return np.array(times), np.array(surv)

fig, ax = plt.subplots(figsize=(9.5, 5.6))
if use_lifelines:
    kmf.fit(df['time'], df['event'])
    ax.step(kmf.survival_function_.index,
            kmf.survival_function_.iloc[:, 0],
            where='post', color=VIZ['series'][0])
else:
    tt, ss = km_estimate(df['time'], df['event'])
    ax.step(tt, ss, where='post', color=VIZ['series'][0])
ax.set_xlabel('Время наблюдения')
ax.set_ylabel('Доля без события (функция выживания)')
ax.set_title('Оценка Каплана — Мейера')
ax.set_ylim(0, 1.02)
fig.tight_layout()
plt.show()

**Как читать этот график.**
- Ось X — время наблюдения (дни).
- Ось Y — доля наблюдений, у которых событие ещё не наступило (функция выживания, от 0 до 1).
- Каждый вертикальный шаг вниз — момент, когда у одного или нескольких объектов произошло событие.
- Горизонтальные участки между шагами — периоды, когда событий не было (только цензурирование).
- Значение кривой в точке t = 200 дней означает: «столько-то процентов объектов дожили до 200-го дня без события».

### Шаг 2. Сравнение групп по полу

Строим отдельные кривые Каплана — Мейера для каждой группы (мужчины и женщины). Если кривые расходятся, это указывает на разную скорость наступления события в группах.

**Лог-ранговый критерий** — статистический тест, проверяющий нулевую гипотезу о том, что кривые выживания в группах одинаковы. Малое p-значение (обычно < 0.05) означает, что различие групп статистически значимо, то есть маловероятно возникло случайно.

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 5.6))
for grp, color in zip(sorted(df['sex'].unique()),
                      [VIZ['series'][0], VIZ['series'][2]]):
    sub = df[df['sex'] == grp]
    if use_lifelines:
        kmf.fit(sub['time'], sub['event'])
        ax.step(kmf.survival_function_.index,
                kmf.survival_function_.iloc[:, 0],
                where='post', color=color, label=f'пол = {grp}')
    else:
        tt, ss = km_estimate(sub['time'], sub['event'])
        ax.step(tt, ss, where='post', color=color,
                label=f'пол = {grp}')
ax.set_xlabel('Время наблюдения')
ax.set_ylabel('Доля без события')
ax.set_title('Выживаемость по группам')
ax.set_ylim(0, 1.02)
ax.legend()
fig.tight_layout()
plt.show()

try:
    from lifelines.statistics import logrank_test
    g = sorted(df['sex'].unique())
    a, b = df[df['sex'] == g[0]], df[df['sex'] == g[1]]
    res = logrank_test(a['time'], b['time'],
                       a['event'], b['event'])
    print(f'Лог-ранговый критерий: p = {res.p_value:.4f}')
except Exception as exc:
    print('Лог-ранговый критерий недоступен без lifelines:', exc)

**Как читать этот график.**
- Две кривые разных цветов — две группы (пол 1 и пол 2).
- Если кривые лежат далеко друг от друга на всём протяжении — группы статистически различаются по выживаемости.
- Если кривые пересекаются, предположение о пропорциональности рисков модели Кокса нарушено.
- p-значение лог-рангового критерия, выведенное ниже: если p < 0.05, различие групп не случайно.

### Шаг 3. Модель пропорциональных рисков Кокса

**Модель Кокса** оценивает, как каждая ковариата влияет на мгновенный риск события. Она не требует задавать математическую форму базового риска — только предполагает, что отношение рисков между любыми двумя объектами постоянно во времени (пропорциональность рисков).

Ключевой результат — таблица **коэффициентов** и **отношений рисков** (`exp(coef)`):
- `exp(coef) > 1` — ковариата повышает риск события (ускоряет его наступление).
- `exp(coef) < 1` — ковариата снижает риск (замедляет наступление события).
- `exp(coef) = 1` — ковариата не влияет на риск.

Маленькое p-значение при коэффициенте означает, что влияние ковариаты статистически значимо.

На графике отображаются отношения рисков с 95-процентными доверительными интервалами. Если интервал не пересекает вертикальную линию «1», влияние значимо.

In [ ]:
try:
    from lifelines import CoxPHFitter
    cox = CoxPHFitter()
    cox.fit(df[['time', 'event', 'age', 'sex', 'ecog']],
            duration_col='time', event_col='event')
    summary = cox.summary[['coef', 'exp(coef)', 'p']]
    print('Отношения рисков (exp(coef)):')
    print(summary.round(3))
    hr = cox.summary['exp(coef)']
    lo = cox.summary['exp(coef) lower 95%']
    hi = cox.summary['exp(coef) upper 95%']
    fig, ax = plt.subplots(figsize=(9.0, 4.6))
    ypos = np.arange(len(hr))
    ax.errorbar(hr.values, ypos,
                xerr=[hr.values - lo.values, hi.values - hr.values],
                fmt='o', color=VIZ['series'][0], capsize=4)
    ax.axvline(1.0, color=VIZ['series'][2], linestyle='--')
    ax.set_yticks(ypos)
    ax.set_yticklabels(hr.index)
    ax.set_xlabel('Отношение рисков (95% доверительный интервал)')
    ax.set_title('Влияние ковариат на риск события')
    fig.tight_layout()
    plt.show()
except Exception as exc:
    print('Модель Кокса требует lifelines:', exc)
    print('Установите: %pip install lifelines')

**Как читать график отношений рисков.**
- Каждая строка — одна ковариата.
- Точка — оценка отношения рисков.
- Горизонтальная линия — 95-процентный доверительный интервал.
- Вертикальная пунктирная линия на значении «1» — нулевой эффект.
- Точка правее «1» с интервалом, не пересекающим «1»: ковариата значимо увеличивает риск.
- Точка левее «1» с интервалом, не пересекающим «1»: ковариата значимо снижает риск.

## 5. Интерпретация результата

- **Кривая Каплана — Мейера**: высота кривой в момент времени — доля наблюдений без события к этому моменту. Ступеньки вниз — моменты наступления событий; цензурированные наблюдения не создают ступенек, но уменьшают число объектов под риском.
- **Сравнение групп**: расхождение кривых указывает на разную выживаемость. Малое p-значение лог-рангового критерия означает, что различие статистически значимо.
- **Отношение рисков** в модели Кокса: значение больше единицы — ковариата повышает риск события, меньше единицы — снижает. Если доверительный интервал не пересекает единицу, эффект значим.

Модель Кокса предполагает пропорциональность рисков во времени; это предположение следует проверять (например, по шёнфельдовским остаткам).

## 6. Поэкспериментируйте сами

Поменяйте один параметр и повторно запустите ячейки раздела 4.

**Что менять и что наблюдать:**
- В разделе 3 измените долю цензурирования: найдите строку `censor_time = rng.uniform(50, 800, n)` и замените `800` на `200`. Что произойдёт с кривой Каплана — Мейера при более агрессивном цензурировании?
- Создайте третью группу: добавьте признак `risk_group = (df['ecog'] >= 2).astype(int)` и постройте кривые для `risk_group == 0` и `risk_group == 1`. Есть ли разница в выживаемости?
- Уберите `ecog` из модели Кокса: измените список ковариат на `['age', 'sex']`. Как изменятся оценки эффекта для оставшихся ковариат?

In [ ]:
# --- Шаблон загрузки своих данных ---
# raw = pd.read_csv('my_data.csv')
# df = pd.DataFrame({
#     'time':  raw['время_наблюдения'].astype(float),
#     'event': raw['событие'].astype(int),   # 1 - событие, 0 - цензур.
#     'age':   raw['ковариата_1'].astype(float),
#     'sex':   raw['ковариата_2'].astype(int),
#     'ecog':  raw['ковариата_3'].astype(float),
# })
# print(f'Наблюдений: {len(df)}')
# Далее повторите ячейки раздела 4.

## Краткие выводы

- Анализ выживаемости незаменим, когда отклик — время до события, а часть наблюдений цензурирована. Игнорировать цензурированные наблюдения нельзя.
- Оценка Каплана — Мейера даёт наглядную непараметрическую кривую выживания; лог-ранговый критерий сравнивает группы без предположений о форме кривых.
- Модель Кокса переводит различия в выживаемости в численные отношения рисков для каждой ковариаты: насколько тот или иной фактор ускоряет или замедляет наступление события.
- Ключевое допущение Кокса — пропорциональность рисков — нужно проверять. При нарушении нужны расширенные модели.
- Отношение рисков показывает относительный эффект, а не абсолютный: «риск вдвое выше» не означает «событие наступит скоро» — это зависит от базового уровня риска.